In [10]:
import sys
sys.path.append('../../')
from model_001 import LucaQuadruple_final_dropout, fluProfiler_Config
from tqdm import tqdm
import os
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import load_embedding, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from datetime import datetime
import pickle
from utilities import print_exams

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def list2df(mylist, period):
    merged_rows = []
    for i in range(0, len(mylist), period):
        merged_row = []
        for j in range(period):
            merged_row += mylist[i + j]
        merged_rows.append(merged_row)

    return pd.DataFrame(merged_rows)

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask

device = torch.device('cuda:3')

data_path = '/data/chenyihao/dataset'
Crick_all = pd.read_csv(data_path + '/all.csv')

In [11]:
Crick_all = Crick_all.loc[(Crick_all['seq_a'].str.len() == 566) & (Crick_all['seq_c'].str.len() == 566)]

In [14]:
group_columns = ['seq_a','seq_c', 'serumPassCat', 'virusPassCat']
# new_columns = ['seq_id_a', 'seq_id_c','seq_a', 'seq_c', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'label']
Crick_filt1 = Crick_all[['seq_id_a', 'seq_id_c', 'seq_a', 'seq_c', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'label']].copy()
Crick_final = Crick_filt1.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_c': 'first', 'serumName': 'first', 'virusName': 'first', 'label': 'mean'}).reset_index()
Crick_final.columns = ['seq_a','seq_c', 'serumPassCat', 'virusPassCat', 'seq_id_a', 'seq_id_c', 'serumName', 'virusName', 'label']

virus_strains = Crick_final[['seq_c', 'virusPassCat']].drop_duplicates().reset_index(drop=True)
virus_strains4test = virus_strains.sample(frac=0.1, random_state=42).reset_index(drop=True)

virus_set = set(zip(virus_strains4test['seq_c'], virus_strains4test['virusPassCat']))
mask = Crick_final.apply(lambda row: (row['seq_c'], row['virusPassCat']) in virus_set, axis=1)
test_data = Crick_final[mask]
train_data, valid_data = train_test_split(Crick_final[~mask], test_size=1/9, random_state=42)

In [16]:
import os
from tqdm import tqdm

embedding_df = test_data
# load embedding
sequence_names = pd.concat([embedding_df['seq_id_a'], 
                            embedding_df['seq_id_c']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding", files=sequence_names)
# embeddings = [emb.to(device) for emb in embeddings]
emb_dict = dict(zip(IDs, embeddings))

Loading tensor: 100%|██████████| 703/703 [01:19<00:00,  8.82file/s]


In [27]:
[emb_dict['matrix_' + row['seq_id_a']]]

[tensor([[ 0.2005,  0.1255, -0.0655,  ..., -0.0437, -0.0168,  0.0560],
         [ 0.1986,  0.0179,  0.0273,  ...,  0.0050,  0.0220,  0.1946],
         [ 0.1747,  0.0126, -0.0253,  ..., -0.0288, -0.0813,  0.1722],
         ...,
         [ 0.0724,  0.0434,  0.0892,  ..., -0.0769, -0.0327,  0.0433],
         [ 0.0744,  0.0522,  0.0819,  ..., -0.0296, -0.0977, -0.1295],
         [ 0.0650,  0.1680,  0.1093,  ...,  0.0074, -0.1761, -0.1106]])]

In [44]:
diff_matrix = []
for index, row in test_data.iterrows():
    matrixs_a, masks_a = generate_matrix([emb_dict['matrix_' + row['seq_id_a']]])
    matrixs_c, masks_c = generate_matrix([emb_dict['matrix_' + row['seq_id_c']]])
    diff_matrix.append(matrixs_a - matrixs_c)

In [45]:
prediction = []

for diff_mat in diff_matrix:
    prediction.append(np.linalg.norm(diff_mat[0], ord='fro'))

In [47]:
print_exams(prediction, test_data['label'])

MAE:  26.404996016803292
MSE:  756.2544937171366
pearson correlation:  PearsonRResult(statistic=0.40406909412310793, pvalue=7.635301481859088e-194)
spearman correlation:  SignificanceResult(statistic=0.38772942989724746, pvalue=2.456212711752219e-177)
R2_score:  -193.9821436253211


In [57]:
test_data_virus_emb = [emb_dict['matrix_' + virus] for virus in test_data['seq_id_c']]


In [ ]:
test_data_virus_emb = torch.stack

ValueError: only one element tensors can be converted to Python scalars

In [ ]:
import numpy as np
import umap
from sklearn.cluster import KMeans
import plotly.graph_objects as go


n_samples   = 2000
seq_len     = 128          # 你的真实 seq_len
emb_dim     = 2560
X = np.random.randn(n_samples, seq_len, emb_dim).astype(np.float32)

# ① 对 seq_len 维度做 mean → (2000, 2560)
X_vec = X.mean(axis=1)      # 也可试 max/GAP/attention pooling

# ② UMAP 降维到 3D（n_neighbors 可调）
umap_3d = umap.UMAP(n_components=3,
                    n_neighbors=30,
                    min_dist=0.1,
                    metric='cosine',
                    random_state=42)
emb_3d = umap_3d.fit_transform(X_vec)   # (2000, 3)

# ③ K-means 聚类（k 按需改）
k = 5
labels = KMeans(n_clusters=k, random_state=42).fit_predict(X_vec)

# ④ 3D 交互式散点图
fig = go.Figure()
for i in range(k):
    mask = labels == i
    fig.add_trace(go.Scatter3d(
        x=emb_3d[mask, 0],
        y=emb_3d[mask, 1],
        z=emb_3d[mask, 2],
        mode='markers',
        marker=dict(size=3),
        name=f'cluster {i}'
    ))
fig.update_layout(
    title='UMAP 3D + K-means',
    scene=dict(xaxis_title='UMAP-1',
               yaxis_title='UMAP-2',
               zaxis_title='UMAP-3'),
    width=900, height=700
)
fig.show()